In [1]:
import pandas as pd
import numpy as np
import fuzzywuzzy
from fuzzywuzzy import process
import charset_normalizer

professors = pd.read_csv("../input/pakistan_intellectual_capital.csv")

np.random.seed(0)

In [2]:
professors.head()

,Unnamed: 0,S#,Teacher Name,University Currently Teaching,Department,Province University Located,Designation,Terminal Degree,Graduated from,Country,Year,Area of Specialization/Research Interests,Other Information
0,2,3,Dr. Abdul Basit,University of Balochistan,Computer Science & IT,Balochistan,Assistant Professor,PhD,Asian Institute of Technology,Thailand,NaN,Software Engineering & DBMS,NaN
1,4,5,Dr. Waheed Noor,University of Balochistan,Computer Science & IT,Balochistan,Assistant Professor,PhD,Asian Institute of Technology,Thailand,NaN,DBMS,NaN
2,5,6,Dr. Junaid Baber,University of Balochistan,Computer Science & IT,Balochistan,Assistant Professor,PhD,Asian Institute of Technology,Thailand,NaN,"Information processing, Multimedia mining",NaN
3,6,7,Dr. Maheen Bakhtyar,University of Balochistan,Computer Science & IT,Balochistan,Assistant Professor,PhD,Asian Institute of Technology,Thailand,NaN,"NLP, Information Retrieval, Question Answering...",NaN
4,24,25,Samina Azim,Sardar Bahadur Khan Women's University,Computer Science,Balochistan,Lecturer,BS,Balochistan University of Information Technolo...,Pakistan,2005.0,VLSI Electronics DLD Database,NaN


In [5]:
# get all the unique values in the 'Country' column
countries = professors['Country'].unique()

# sort them alphabetically and then take a closer look
countries = sorted(countries)
countries

[' Germany',
 ' New Zealand',
 ' Sweden',
 ' USA',
 'Australia',
 'Austria',
 'Canada',
 'China',
 'Finland',
 'France',
 'Greece',
 'HongKong',
 'Ireland',
 'Italy',
 'Japan',
 'Macau',
 'Malaysia',
 'Mauritius',
 'Netherland',
 'New Zealand',
 'Norway',
 'Pakistan',
 'Portugal',
 'Russian Federation',
 'Saudi Arabia',
 'Scotland',
 'Singapore',
 'South Korea',
 'SouthKorea',
 'Spain',
 'Sweden',
 'Thailand',
 'Turkey',
 'UK',
 'USA',
 'USofA',
 'Urbana',
 'germany']

In [6]:
# convert to lower case
professors['Country'] = professors['Country'].str.lower()
# remove trailing white spaces
professors['Country'] = professors['Country'].str.strip()

In [8]:
# get all the unique values in the 'Country' column
countries = professors['Country'].unique()

# sort them alphabetically and then take a closer look
countries = sorted(countries)
countries

['australia',
 'austria',
 'canada',
 'china',
 'finland',
 'france',
 'germany',
 'greece',
 'hongkong',
 'ireland',
 'italy',
 'japan',
 'macau',
 'malaysia',
 'mauritius',
 'netherland',
 'new zealand',
 'norway',
 'pakistan',
 'portugal',
 'russian federation',
 'saudi arabia',
 'scotland',
 'singapore',
 'south korea',
 'southkorea',
 'spain',
 'sweden',
 'thailand',
 'turkey',
 'uk',
 'urbana',
 'usa',
 'usofa']

**Fuzzy matching**: The process of automatically finding text strings that are very similar to the target string. In general, a string is considered "closer" to another one the fewer characters you'd need to change if you were transforming one string into another. So "apple" and "snapple" are two changes away from each other (add "s" and "n") while "in" and "on" and one change away (rplace "i" with "o"). You won't always be able to rely on fuzzy matching 100%, but it will usually end up saving you at least a little time.

Fuzzywuzzy returns a ratio given two strings. The closer the ratio is to 100, the smaller the edit distance between the two strings. Here, we're going to get the ten strings from our list of cities that have the closest distance to "south korea".

In [9]:
# get the top 10 closest matches to "south korea"
matches = fuzzywuzzy.process.extract("south korea", countries, limit=10, scorer=fuzzywuzzy.fuzz.token_sort_ratio)

# take a look at them
matches

[('south korea', 100),
 ('southkorea', 48),
 ('saudi arabia', 43),
 ('norway', 35),
 ('austria', 33),
 ('ireland', 33),
 ('pakistan', 32),
 ('portugal', 32),
 ('scotland', 32),
 ('australia', 30)]

In [10]:
# function to replace rows in the provided column of the provided dataframe
# that match the provided string above the provided ratio with the provided string
def replace_matches_in_column(df, column, string_to_match, min_ratio = 47):
    # get a list of unique strings
    strings = df[column].unique()

    # get the top 10 closest matches to our input string
    matches = fuzzywuzzy.process.extract(string_to_match, strings,
                                         limit=10, scorer=fuzzywuzzy.fuzz.token_sort_ratio)

    # only get matches with a ratio > 90
    close_matches = [matches[0] for matches in matches if matches[1] >= min_ratio]

    # get the rows of all the close matches in our dataframe
    rows_with_matches = df[column].isin(close_matches)

    # replace all rows with close matches with the input matches
    df.loc[rows_with_matches, column] = string_to_match

    # let us know the function's done
    print("All done!")

In [11]:
# use the function we just wrote to replace close matches to "south korea" with "south korea"
replace_matches_in_column(df=professors, column='Country', string_to_match="south korea")

All done!


In [12]:
# get all the unique values in the 'Country' column
countries = professors['Country'].unique()

# sort them alphabetically and then take a closer look
countries = sorted(countries)
countries

['australia',
 'austria',
 'canada',
 'china',
 'finland',
 'france',
 'germany',
 'greece',
 'hongkong',
 'ireland',
 'italy',
 'japan',
 'macau',
 'malaysia',
 'mauritius',
 'netherland',
 'new zealand',
 'norway',
 'pakistan',
 'portugal',
 'russian federation',
 'saudi arabia',
 'scotland',
 'singapore',
 'south korea',
 'spain',
 'sweden',
 'thailand',
 'turkey',
 'uk',
 'urbana',
 'usa',
 'usofa']

# exercise